## Setup Colab and sync


In [ ]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os
# Api key setting
os.environ['patentsview_api_key'] = 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'  # Replace with your actual key

Cloning into 'bert_for_patents'...
remote: Enumerating objects: 879, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 879 (delta 14), reused 13 (delta 13), pack-reused 854 (from 4)
Receiving objects: 100% (879/879), 159.23 MiB | 13.64 MiB/s, done.
Resolving deltas: 100% (479/479), done.


In [ ]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
Already up to date.


In [1]:
#@title Run setup file to get environment
!python "/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py"

import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

python3: can't open file '/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py': [Errno 2] No such file or directory


NameError: name 'os' is not defined

In [ ]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


Set precision

In [ ]:
#@title precision
import api.fetch_patents
import api.find_similar_patents
import importlib
import api.bertembeddings as be
import tensorflow as tf
tf.config.optimizer.set_jit(True)
from tensorflow.keras import mixed_precision

# Set the global policy to mixed precision
mixed_precision.set_global_policy('mixed_float16')

# Verify the policy
print("Compute dtype:", mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", mixed_precision.global_policy().variable_dtype)

Compute dtype: float16
Variable dtype: float32


## Main routine to get the closest patents

In [ ]:
#@title Process assignees and find closest patents of combined DF
# Import necessary functions
import importlib
import api.colab_fncs as col
import api.find_similar_patents as fsp
import api.fetch_patents as fp
importlib.reload(col)
importlib.reload(fsp)
importlib.reload(fp)
import pandas as pd
from datetime import datetime
import time

# Get the data
df = pd.read_excel("/content/bert_for_patents/05 Analysis/01 Main/00 Python data/sdc_google_combined.xlsx")

# Ensure 'Date Effective' column is in datetime format and handle potential errors
try:
    df['Date Effective'] = pd.to_datetime(df['Date Effective'], errors='coerce')  # Convert to datetime, invalid parsing will be NaT
    df['Date Effective'] = df['Date Effective'].dt.strftime('%Y-%m-%d')  # Format to desired string format
except KeyError:
    print("Column 'Date Effective' not found in DataFrame.")
    # Handle the KeyError, e.g., exit or provide alternative logic
except Exception as e:
    print(f"Error processing 'Date Effective' column: {e}")
    # Handle other exceptions as needed

start_time_field = time.time()
start_time_embeddings = time.time()

# Run the routine
for index, row in df.iterrows():
        if index < 190:
          continue
        # Report progress
        print(f"Handling row {index} of {len(df)}")

        assignee_name = row['Assignees']
        date_effective = row['Date Effective']

        # Try to load previously saved patents
        patents_before = col.load_patents_from_drive(assignee_name, f"{date_effective}_before")
        patents_after = col.load_patents_from_drive(assignee_name, f"{date_effective}_after")

        if patents_before is None or patents_after is None:
            # Get patents if not previously saved
            patents_before_after = api.fetch_patents.get_patents(assignee_name, date_effective)
            if patents_before_after is None:
              print("Likely Internal query error")
              continue
            patents_before = patents_before_after[0]
            patents_after = patents_before_after[1]
            print("Not available in Drive")

            # Save patents for future use
            col.save_patents_to_drive(patents_before, assignee_name, f'{date_effective}_before')
            col.save_patents_to_drive(patents_after, assignee_name, f'{date_effective}_after')
        else:
            print(f"Patents for {assignee_name} at {date_effective} loaded from Drive.")

        print(f"{len(patents_before)} patents before; {len(patents_after)} patents after for Assignee : {assignee_name}")

        # Now find the closest patent for each in patents_before
        counter = 0

        for patent in patents_before:
            counter += 1
            # Check if the patent has already been processed
            if patent.closest_patent is not None:
                continue
            print(f"Currently doing patent ID: {patent.patent_id}")

            closest_patent, distance_cs, distance_eu = api.find_similar_patents.find_closest_patent(patent, group_only=True, batch_size=32, filter_tfidf=True)
            if closest_patent is None:
                print("Error happened: skipping this patent")
                continue
            # Add closest patent information to the patent object
            patent.closest_patent = closest_patent
            patent.cosine_similarity_with_closest_patent = distance_cs
            patent.euclidean_distance_to_closest_patent = distance_eu

            # Change: Ensure the save function is called every 20 minutes
            if time.time() - start_time_field >= 30 * 60:  # 20 minutes in seconds
              print("Saving Field dictionary")
              api.find_similar_patents.save_field_dict_every_20_minutes(field_dict_folder='/content/drive/MyDrive/PhD Data/04 Field dictionaries', default_file="/content/drive/MyDrive/PhD Data/04 Field dictionaries/Field Dict default file.pkl")
              start_time_field = time.time()

            # Change: Ensure the save function is called every 20 minutes
            if time.time() - start_time_embeddings >= 30 * 60:  # 20 minutes in seconds
              api.find_similar_patents.save_patents_every_20_minutes(checked_patents_folder='/content/drive/MyDrive/PhD Data/01 CLS Embeddings', default_file="/content/drive/MyDrive/PhD Data/01 CLS Embeddings/CheckedPatents_CLSonly_2024.10.21_00:45.pkl")
              start_time_embeddings = time.time()


            # Save updated patent information every 100 patent
            if counter % 100 == 0:
              col.save_patents_to_drive(patents_before, assignee_name, f'{date_effective}_before')
            print(f"{counter}/{len(patents_before)} patents processed")

      wadee


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install dill

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 11.7 MB/s eta 0:00:00


In [ ]:
#@title Get available embeddings
import dill as pickle
with open("/content/drive/MyDrive/PhD Data/01 CLS Embeddings/CheckedPatents_CLSonly_2024.11.11_00:06.pkl", 'rb') as f:
    available_embeddings = pickle.load(f)

len(available_embeddings)

2077783

In [ ]:
print('10009178' in available_embeddings)
type(available_embeddings['10009178'])

True


numpy.ndarray

In [ ]:
#@ Get the IDs of Patents which have been fed to BERT already
import csv

# Specify the output CSV file name
output_csv_file = "fed_patents.csv"

# Write the keys to the CSV file
with open(output_csv_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    for key in available_embeddings.keys():
        writer.writerow([key])

print(f"Keys have been written to {output_csv_file}.")

Keys have been written to fed_patents.csv.


In [ ]:
from google.colab import files
files.download(output_csv_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title remove already checked patents

not_checked = []
for patent in t:
  if patent.patent_id not in available_embeddings:
    not_checked.append(patent)

len(not_checked)

823641

In [ ]:
#@title Feed not checked patents to BERT and save their ID-Embedding pairs
import api.bertembeddings as be
import importlib
#importlib.reload(be)
batch_size = 32

batched_patents = [not_checked[i:i + batch_size] for i in range(0, len(not_checked), batch_size)]

counter = 0

#embeddings_to_be_saved = {}

for batch in batched_patents:
  abstracts = [d.abstract for d in batch]
  ids = [d.patent_id for d in batch if d is not None]

  if len(ids) < batch_size:
    print('Skipping this batch -- None Patent ID somewhere in the batch')
    continue
  counter += batch_size
  print(counter)
  emb = be.get_embd_of_whole_abstract(abstracts, has_context_token = True)
  for i in range(len(ids)):
    embeddings_to_be_saved[ids[i]] = emb[i]


## Data regularization - involving some of the steps above ☁

In [ ]:
#@title utils
from api.patent import Patent
import dill as pickle

def load(path):
    with open(path, mode = "rb") as f:
        file = pickle.load(f)
    return file
def save(path, file):
    with open(path, mode = "wb") as f:
        pickle.dump(file, f)

import re
import os

import re

def get_assignee_name(filename):
  """
  Extracts the assignee name from the filename.

  Args:
    filename: The filename to extract the assignee name from.

  Returns:
    The assignee name, or None if it could not be extracted.
  """
  match = re.search(r"^(.*?)_", filename)  # Matches everything before the first underscore
  if match:
    return match.group(1)  # Returns the matched group
  else:
    return None  # Returns None if no match is found

def extract_date(filename):
  """Extracts the date in YYYY-MM-DD format from the filename."""
  match = re.search(r"(\d{4}-\d{2}-\d{2})", filename)  # Regular expression to find the date pattern
  if match:
    effective_date = match.group(1)  # Get the matched date string
    return effective_date
  else:
    return None  # Return None if no date is found

def update_patent_groups(old_patents, new_patents):
    """Compares two lists of patents and updates tech_field_group, subgroup,
    tech_field_group_id, and subgroup_id if different in the new list.

    Args:
        old_patents: A list of Patent objects.
        new_patents: A list of Patent objects with potentially updated group information.
    """
    old_patent_dict = {patent.patent_id: patent for patent in old_patents}

    changed_indices = []

    for i, new_patent in enumerate(new_patents):
        if new_patent.patent_id in old_patent_dict:
            old_patent = old_patent_dict[new_patent.patent_id]
            old_patent_index = old_patents.index(old_patent)  # Get the index of the old patent in the original list

            # Update tech_field_group and tech_field_group_id
            if old_patent.tech_field_group != new_patent.tech_field_group or \
               old_patent.tech_field_group_id != new_patent.tech_field_group_id:
                old_patent.tech_field_group = new_patent.tech_field_group
                old_patent.tech_field_group_id = new_patent.tech_field_group_id
                print(f"Updated tech_field_group for patent {new_patent.patent_id}")
                changed_indices.append(old_patent_index)

            # Update subgroup and subgroup_id
            if old_patent.tech_field_subgroup != new_patent.tech_field_subgroup or \
               old_patent.tech_field_subgroup_id != new_patent.tech_field_subgroup_id:
                old_patent.tech_field_subgroup = new_patent.tech_field_subgroup
                old_patent.tech_field_subgroup_id = new_patent.tech_field_subgroup_id
                print(f"Updated subgroup for patent {new_patent.patent_id}")

    return changed_indices

def check_regular_patent_content(patent):
    """Checks if a patent object has all required data.
    Raises AssertionError with a descriptive message if data is missing.

    Args:
        patent: A Patent object.
    """
    try:
        assert patent.closest_patent is not None, f"Patent {patent.patent_id}: closest_patent is missing"
        assert patent.cosine_similarity_with_closest_patent is not None, f"Patent {patent.patent_id}: cosine_similarity_with_closest_patent is missing"
        assert patent.euclidean_distance_to_closest_patent is not None, f"Patent {patent.patent_id}: euclidean_distance_to_closest_patent is missing"
        assert patent.patent_id is not None, f"Patent: patent_id is missing"
        assert patent.abstract is not None, f"Patent {patent.patent_id}: abstract is missing"
        assert patent.date_application is not None, f"Patent {patent.patent_id}: date_application is missing"
        assert patent.patent_embedding is not None, f"Patent {patent.patent_id}: patent_embedding is missing"
        assert patent.tech_field_group is not None, f"Patent {patent.patent_id}: tech_field_group is missing"
        assert patent.tech_field_group_id is not None, f"Patent {patent.patent_id}: tech_field_group_id is missing"
        assert patent.tech_field_subgroup is not None, f"Patent {patent.patent_id}: tech_field_subgroup is missing"
    except AssertionError as e:
        print(e)  # Print the error message
        # You can add other error handling logic here, like logging or raising a custom exception






In [ ]:
#@title Patent regularization - making sure that every patent has a closest patent
import importlib
import api.colab_fncs as col
import api.find_similar_patents as fsp
import api.fetch_patents as fp
importlib.reload(col)
importlib.reload(fsp)
importlib.reload(fp)
import pandas as pd
from datetime import datetime
import time

SAVE_DIR = '/content/drive/My Drive/PhD Data/04 Patents with pairs (group checked, regularized, only before)'

start_time_field = time.time()
start_time_embeddings = time.time()


# Load patent objects
patent_folder = r"/content/drive/MyDrive/PhD Data/03 Patents with pairs (group checked)"
acquired_patent_paths = [os.path.join(patent_folder, o) for o in os.listdir(patent_folder) if o.endswith("before.pkl")]


## Patents regularization
for path in acquired_patent_paths:


    counter = 0

    # Extract date and name
    effective_date = extract_date(path)
    filename = os.path.basename(path)
    new_filepath = os.path.join(SAVE_DIR, filename)

    ## Skip if already regularizes
    if os.path.exists(new_filepath) and os.path.getsize(new_filepath) > 10:
        print(f"Skipping {filename} as it has already been regularized")
        continue

    # Load patents
    patents = load(path)

    print(f"Size of patent objects is: {len(patents)}")
    print(f'Currently doing {filename}')

    # Regularize patents with tech fields misspecified
    ## Send the API call to get the data
    before, after = api.fetch_patents.get_patents(get_assignee_name(filename), effective_date)
    if before is None or after is None:
        print("Likely Internal query error")
        continue

    ## Update patent data
    changed_indices = update_patent_groups(patents, before)

    clean_patents = []

    for i, patent in enumerate(patents):


        if i in changed_indices:
          print("Found an updated patent... updating all the info")

        if before is None or after is None:
            print("Likely Internal query error")
            continue



        if patent.closest_patent is None or i in changed_indices:



            counter += 1

            if patent.tech_field_group_id is None:
                print("No Tech field group: skipping this patent")
                continue

            if patent.abstract is None:
                print("No abstract: skipping this patent")
                continue

            print('Currently doing ID: ', patent.patent_id)

            closest_patent, distance_cs, distance_eu = api.find_similar_patents.find_closest_patent(patent, group_only=True, batch_size=32, filter_tfidf=True)
            if closest_patent is None:
                print("Error happened: skipping this patent")
                continue

            print("None closest Patent!!!!")


            # Add closest patent information to the patent object
            patent.closest_patent = closest_patent
            patent.cosine_similarity_with_closest_patent = distance_cs
            patent.euclidean_distance_to_closest_patent = distance_eu

            ## ensure that everything is regular
            check_regular_patent_content(patent)




        # Change: Ensure the save function is called every 20 minutes
        if time.time() - start_time_field >= 90 * 60:  # 20 minutes in seconds
          print("Saving Field dictionary")
          api.find_similar_patents.save_field_dict_every_20_minutes(field_dict_folder='/content/drive/MyDrive/PhD Data/04 Field dictionaries', default_file='/content/drive/MyDrive/PhD Data/04 Field dictionaries/Combined Field Dict.pkl')
          start_time_field = time.time()

        # Change: Ensure the save function is called every 20 minutes
        if time.time() - start_time_embeddings >= 90 * 60:  # 20 minutes in seconds
          api.find_similar_patents.save_patents_every_20_minutes(checked_patents_folder='/content/drive/MyDrive/PhD Data/01 CLS Embeddings', default_file="/content/drive/MyDrive/PhD Data/01 CLS Embeddings/CheckedPatents_CLSonly_2024.10.24_05:22.pkl")
          start_time_embeddings = time.time()

        # Save updated patent information every 100 patent
        if counter % 100 == 0:
          with open(new_filepath, 'wb') as f:
            pickle.dump(clean_patents, f)

        print(f"{i}/{len(patents)} patents processed")
        clean_patents.append(patent)

    # Save updated patent information
    with open(new_filepath, 'wb') as f:
        pickle.dump(clean_patents, f)

    regularized_filenames.append(filename)

In [ ]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 492 bytes | 246.00 KiB/s, done.
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
   532ff72..0ef86ad  master     -> origin/master
Updating 532ff72..0ef86ad
Fast-forward
 05 Analysis/01 Main/api/citations.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [ ]:
## Define folder for the patents before
folder_path = r"/content/drive/MyDrive/PhD Data/04 Patents with pairs (group checked, regularized, only before)"
SAVE_DIR = '/content/drive/My Drive/PhD Data/05 Patents with pairs (group checked, regularized, only before)'

# Get a list of all files in the folder
files = os.listdir(folder_path)

import api.citations as cit
from datetime import datetime
import importlib
importlib.reload(cit)

all_patents = []
all_skipped_patents = []

# Iterate over each file
for filename in files:

  patents = load(os.path.join(folder_path, filename))
  if len(patents) == 0:
    continue

  print(f"Currently doing {filename}")

  patents, skipped_patents = cit.regularize_patents(patents)
  print(f"Skipped {len(skipped_patents)} patents")


  for patent in patents:
    patent.date_acquired = datetime.strptime(extract_date(filename), "%Y-%m-%d")
  all_patents.append(patents)
  all_skipped_patents.append(skipped_patents)





Streaming output truncated to the last 5000 lines.
Generating embedding for closest patent of patent 8320575
There are 1 patents being fed into BERT
It took 0.02732 seconds to get the embeddings of the input.
Processed patent 613/1414
Generating embedding for closest patent of patent 8321222
There are 1 patents being fed into BERT
It took 0.02498 seconds to get the embeddings of the input.
Processed patent 614/1414
Generating embedding for closest patent of patent 8321277
There are 1 patents being fed into BERT
It took 0.02896 seconds to get the embeddings of the input.
Processed patent 615/1414
Generating embedding for closest patent of patent 8326629
There are 1 patents being fed into BERT
It took 0.02535 seconds to get the embeddings of the input.
Processed patent 616/1414
Generating embedding for closest patent of patent 8326653
There are 1 patents being fed into BERT
It took 0.02766 seconds to get the embeddings of the input.
Processed patent 617/1414
Generating embedding for clos

In [ ]:
print(len(all_patents))
print(len(all_skipped_patents))

161
161


In [ ]:
all = [patent for patents in all_patents for patent in patents]
print(len(all))

5292


In [ ]:
skipped = [patent for patents in all_skipped_patents for patent in patents]
print(len(skipped))

0


In [ ]:
SAVE_DIR = '/content/drive/My Drive/PhD Data/05 Patents with pairs (group checked, regularized, all)'

with open(SAVE_DIR + '/all_patents.pkl', 'wb') as f:
    pickle.dump(all, f)

## Add Motorola Mobility patents

In [ ]:
import importlib
import api.colab_fncs as col
import api.find_similar_patents as fsp
import api.fetch_patents as fp
importlib.reload(col)
importlib.reload(fsp)
importlib.reload(fp)
import pandas as pd
from datetime import datetime
import time

# Get the data
df = pd.read_excel("/content/bert_for_patents/05 Analysis/01 Main/00 Python data/sdc_google_combined.xlsx")

# Ensure 'Date Effective' column is in datetime format and handle potential errors
try:
    df['Date Effective'] = pd.to_datetime(df['Date Effective'], errors='coerce')  # Convert to datetime, invalid parsing will be NaT
    df['Date Effective'] = df['Date Effective'].dt.strftime('%Y-%m-%d')  # Format to desired string format
except KeyError:
    print("Column 'Date Effective' not found in DataFrame.")
    # Handle the KeyError, e.g., exit or provide alternative logic
except Exception as e:
    print(f"Error processing 'Date Effective' column: {e}")
    # Handle other exceptions as needed

start_time_field = time.time()
start_time_embeddings = time.time()

# Run the routine
for index, row in df.iterrows():
        if index < 190:
          continue
        # Report progress
        print(f"Handling row {index} of {len(df)}")

        assignee_name = row['Assignees']
        date_effective = row['Date Effective']
        print(type(assignee_name), assignee_name)
        print(type(date_effective), date_effective)

{'X-Api-Key': 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'}
Handling row 190 of 211
<class 'str'> Affirmed Networks, Inc.
<class 'str'> 2020-04-22
Handling row 191 of 211
<class 'str'> NextVR Inc.
<class 'str'> 2020-05-13
Handling row 192 of 211
<class 'str'> GIPHY, INC.
<class 'str'> 2020-05-14
Handling row 193 of 211
<class 'str'> American Well Corporation
<class 'str'> 2020-08-23
Handling row 194 of 211
<class 'str'> ZeniMax Media Inc.
<class 'str'> 2021-03-08
Handling row 195 of 211
<class 'str'> Actifio, Inc.
<class 'str'> 2020-12-13
Handling row 196 of 211
<class 'str'> THE MARSDEN GROUP
<class 'str'> 2021-03-15
Handling row 197 of 211
<class 'str'> Nuance Communications, Inc.
<class 'str'> 2022-03-03
Handling row 198 of 211
<class 'str'> PROVINO TECHNOLOGIES, INC.
<class 'str'> 2021-05-12
Handling row 199 of 211
<class 'str'> ReFirm Labs, Inc.
<class 'str'> 2021-06-01
Handling row 200 of 211
<class 'str'> WICKR INC.
<class 'str'> 2021-06-24
Handling row 201 of 211
<class 'str'> A

In [ ]:
#@title Process assignees and find closest patents of combined DF
# Import necessary functions
import importlib
import api.colab_fncs as col
import api.find_similar_patents as fsp
import api.fetch_patents as fp
importlib.reload(col)
importlib.reload(fsp)
importlib.reload(fp)
import pandas as pd
from datetime import datetime
import time


start_time_field = time.time()
start_time_embeddings = time.time()


assignee_name = "Motorola Mobility LLC"
date_effective = "2012-05-21"

# Try to load previously saved patents
patents_before = col.load_patents_from_drive(assignee_name, f"{date_effective}_before")
patents_after = col.load_patents_from_drive(assignee_name, f"{date_effective}_after")

if patents_before is None or patents_after is None:
    # Get patents if not previously saved
    patents_before_after = api.fetch_patents.get_patents(assignee_name, date_effective)
    if patents_before_after is None:
      print("Likely Internal query error")

    patents_before = patents_before_after[0]
    patents_after = patents_before_after[1]
    print("Not available in Drive")

    # Save patents for future use
    col.save_patents_to_drive(patents_before, assignee_name, f'{date_effective}_before')
    col.save_patents_to_drive(patents_after, assignee_name, f'{date_effective}_after')
else:
    print(f"Patents for {assignee_name} at {date_effective} loaded from Drive.")

print(f"{len(patents_before)} patents before; {len(patents_after)} patents after for Assignee : {assignee_name}")


{'X-Api-Key': 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'}
/content/drive/My Drive/PhD Data/04 Patents with pairs (group checked, regularized, only before)/Motorola Mobility LLC_2012-05-21_before.pkl
/content/drive/My Drive/PhD Data/04 Patents with pairs (group checked, regularized, only before)/Motorola Mobility LLC_2012-05-21_after.pkl
https://api.patentsview.org/patents/query?o={"page":1,"per_page":10000}&q={"assignee_organization": "Motorola Mobility LLC"}&f=["cpc_subgroup_id", "cpc_group_id", "cpc_group_title", "cpc_subgroup_title", "cpc_sequence" ,"app_date", "patent_date","patent_id", "patent_date","cpc_category","assignee_country", "assignee_organization", "assignee_id", "patent_num_cited_by_us_patents", "patent_abstract", "citedby_patent_id", "citedby_patent_title", "citedby_patent_date"]
Size of patents before:  1306 Size of patents after:  0
Not available in Drive
1306 patents before; 0 patents after for Assignee : Motorola Mobility LLC


In [ ]:
print(patents_before[5])

Patent(patent_id='7801065', abstract='An access point utilizes information (11) as provided by various subscriber units to provide a message (12) (as included with, for example, a beacon transmission) that reflects near term likely utilization of a shared communication resource. A subscriber unit (20) can then utilize that information to schedule its own monitoring activity. This, in turn, permits the subscriber unit to similarly schedule power saving activities to accommodate this reception schedule.', forward_citations=4, date_application=datetime.datetime(2003, 11, 25, 0, 0), date_granted=datetime.datetime(2010, 9, 21, 0, 0), date_acquired=None, tech_field_group='inventional', tech_field_group_id='H04W', tech_field_subgroup='Supervisory, monitoring or testing arrangements-Scheduling measurement reports ; ; Arrangements for measurement reports', tech_field_subgroup_id='H04W24/10', assignee_organization='Motorola Mobility LLC', assignee_country='US', assignee_id='4479', citedby_patent

In [ ]:
from datetime import datetime
def get_current_timestamp():
    return datetime.now().strftime('%Y.%m.%d_%H:%M')
# Now find the closest patent for each in patents_before
counter = 0

for patent in patents_before:
    counter += 1
    # Check if the patent has already been processed
    if patent.closest_patent is not None:
        continue
    print(f"Currently doing patent ID: {patent.patent_id}")

    closest_patent, distance_cs, distance_eu = api.find_similar_patents.find_closest_patent(patent, group_only=True, batch_size=32, filter_tfidf=True)
    if closest_patent is None:
        print("Error happened: skipping this patent")
        continue
    # Add closest patent information to the patent object
    patent.closest_patent = closest_patent
    patent.cosine_similarity_with_closest_patent = distance_cs
    patent.euclidean_distance_to_closest_patent = distance_eu

    check_regular_patent_content(patent)

    # Change: Ensure the save function is called every 20 minutes
    if time.time() - start_time_field >= 30 * 60:  # 20 minutes in seconds
      print("Saving Field dictionary")
      api.find_similar_patents.save_field_dict_every_20_minutes(field_dict_folder='/content/drive/MyDrive/PhD Data/04 Field dictionaries', default_file="/content/drive/MyDrive/PhD Data/04 Field dictionaries/Field Dict default file.pkl")
      start_time_field = time.time()

    # Change: Ensure the save function is called every 20 minutes
    if time.time() - start_time_embeddings >= 30 * 60:  # 20 minutes in seconds
      api.find_similar_patents.save_patents_every_20_minutes(checked_patents_folder='/content/drive/MyDrive/PhD Data/01 CLS Embeddings', default_file="/content/drive/MyDrive/PhD Data/01 CLS Embeddings/CheckedPatents_CLSonly_2024.10.21_00:45.pkl")
      start_time_embeddings = time.time()


    # Save updated patent information every 100 patent
    if counter % 100 == 0:
      timestamp = get_current_timestamp()
      col.save_patents_to_drive(patents_before, assignee_name, f'{date_effective}_before_{timestamp}')
    print(f"{counter}/{len(patents_before)} patents processed")



Streaming output truncated to the last 5000 lines.
Currently doing TF-IDF
There were 6971 files in total. After TF-IDF filtering, there are 3769 patents left now.
There are 32 patents being fed into BERT
It took 0.03124 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.03114 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.03106 seconds to get the embeddings of the input.
There are 30 patents being fed into BERT
It took 0.03072 seconds to get the embeddings of the input.
There are 1 patents being fed into BERT
It took 0.02788 seconds to get the embeddings of the input.
Size of own (1, 1024)
Size of against (3769, 1024)
https://api.patentsview.org/patents/query?o={"page":1,"per_page":10000}&q={"patent_id":9131233}&f=["cpc_category", "cpc_subgroup_id", "cpc_group_id", "cpc_group_title", "cpc_subgroup_title", "app_date", "patent_date","patent_id", "patent_abstract", "assignee_country", "assignee_org

In [ ]:
col.save_patents_to_drive(patents_before, assignee_name, f'{date_effective}_before_0135')

In [ ]:
with open('/content/drive/MyDrive/PhD Data/04 Patents with pairs (group checked, regularized, only before)/Motorola Mobility LLC_2012-05-21_before_0135.pkl', 'rb') as f:
    op = pickle.load(f)

In [ ]:
op[659]

Patent(patent_id='8379601', abstract='A method for selective use of control channel element (CCE)-based implicit pointing. The method includes the step of determining whether a number of multiple user elements (UE) within a multi-user multiple-input multiple-output (MU-MIMO) group is greater than the number of resource blocks allocated to the MU-MIMO group. If the number of UEs in the MU-MIMO group is greater than the number of resource blocks allocated to the MU-MIMO group, the method further includes transmitting to each of the UEs of the MU-MIMO group acknowledgements on acknowledgement channels within a first acknowledgement bank and acknowledgements on acknowledgement channels within a second acknowledgement bank. A first portion of the UEs of the MU-MIMO group receives the acknowledgements on the acknowledgement channels within the first acknowledgement bank and a second portion of the UEs of MU-MIMO group receives the acknowledgements on the acknowledgement channels within the s

# Now combining patents with their abstracts from bulk data in Drive

In [ ]:
#@title Functions to feed abstracts in bulk
import pandas as pd
import numpy as np
import time
import dill as pickle  # to save and load large objects
from tqdm import tqdm  # for progress bars
import api.bertembeddings as be

# Function to load CSV and batch data
def load_and_batch_data(csv_file, batch_size):
    # Load the CSV file into a pandas DataFrame
    data = pd.read_csv(csv_file)

    # Create batches of abstracts
    for i in range(0, len(data), batch_size):
        batch = data.iloc[i:i+batch_size]
        yield batch['patent_id'].tolist(), batch['abstract'].tolist()

# Function to save embeddings dictionary
def save_embeddings(embeddings_dict, output_file):
    with open(output_file, 'wb') as f:
        pickle.dump(embeddings_dict, f)

# Main function to process and save embeddings
def process_and_save_embeddings(csv_file, batch_size, save_interval=3600, output_file="embeddings.pkl"):
    embeddings_dict = {}
    start_time = time.time()
    processed_count = 0

    # Load and process data in batches
    for patent_ids, abstracts in tqdm(load_and_batch_data(csv_file, batch_size)):
        # Get embeddings for the current batch
        embeddings = be.get_embd_of_whole_abstract(abstracts)

        # Map embeddings to their respective patent IDs
        for patent_id, embedding in zip(patent_ids, embeddings):
            embeddings_dict[patent_id] = embedding
            print(type(embedding))
            print(embedding.shape)

        processed_count += len(patent_ids)

        # Save embeddings periodically based on the save interval
        if time.time() - start_time >= save_interval:
            save_embeddings(embeddings_dict, output_file)
            print(f"Saved {processed_count} embeddings so far.")
            start_time = time.time()

    # Save the final embeddings dictionary
    save_embeddings(embeddings_dict, output_file)
    print(f"All embeddings processed and saved. Total patents: {processed_count}.")

# Example usage
csv_file = pd.read_csv('/content/drive/MyDrive/PhD Data/06 Patents to be fed to BERT/Patents to be fed to BERT.csv', dtype={'patent_id': str})  # Path to your CSV file
batch_size = 32  # Adjust batch size based on available memory
output_file = "/content/drive/MyDrive/PhD Data/patent_embeddings.pkl"

process_and_save_embeddings(csv_file, batch_size, save_interval=3600, output_file=output_file)

0it [00:00, ?it/s]


TypeError: argument of type 'method' is not iterable

In [ ]:
print(len(patents))

9075421
